# Notebook 04 -- Análisis de Clústeres y Clasificadores

---


## Objetivo

Validar que los clusters capturan patrones reales vs. calendario. Entrenar clasificadores multi-configuración para predecir cluster assignment desde features.


## Entrada

data/processed/features.parquet + asignaciones de cluster


## Salida

Modelos clasificadores entrenados (LogReg, RF, SVM, GradBoost) + reportes de accuracy/ROC-AUC


## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

## Imports y Configuración

In [ ]:
import os
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.metrics import (
    cohen_kappa_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED = 42
N_JOBS = -1

# ---- Plot style ----
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'serif',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_palette('colorblind')

np.random.seed(RANDOM_SEED)
print("Configuracion cargada.")

## Carga de datos

In [ ]:
# Cargar variables
features_path = os.path.join(DATA_DIR, "processed", "features.parquet")
df_features = pd.read_parquet(features_path)
# Convertir el índice 'datetime' en una columna 'timestamp'
df_features = df_features.reset_index().rename(columns={'datetime': 'timestamp'})
print(f"Forma de las variables: {df_features.shape}")
print(f"Columnas de las variables: {list(df_features.columns)}")

# Cargar asignaciones de clústeres
clusters_path = os.path.join(DATA_DIR, "clustering", "cluster_assignments.parquet")
df_clusters = pd.read_parquet(clusters_path)
# Cada asignación trae su fecha (columna datetime, ver notebook 03). Reconstruirla
# por posición la desplazaba 48 h, porque el clustering descarta la primera semana ISO incompleta.
df_clusters = df_clusters.rename(columns={'datetime': 'timestamp'})

print(f"\nForma de los clústeres: {df_clusters.shape}")
print(f"Columnas de los clústeres: {list(df_clusters.columns)}")

# Asegurar que timestamp sea datetime en ambos (ahora garantizado por creación explícita)
df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])
df_clusters['timestamp'] = pd.to_datetime(df_clusters['timestamp'])

# Unir por timestamp
df = df_features.merge(df_clusters, on='timestamp', how='inner')
print(f"\nForma combinada: {df.shape}")
print(f"Rango de fechas: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Extraer variables de calendario para el análisis
df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0=Lun, 6=Dom
df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour

In [ ]:
# Identificar cluster columnas
cluster_cols = [c for c in df.columns if 'cluster' in c.lower()]
print(f"\nCluster columnas encontradas: {cluster_cols}")
df.head()

In [ ]:
df = df[~df["timestamp"].dt.year.isin([2020, 2021])]

In [ ]:
# Identificar las columnas de asignación de clúster para tipos de día y de semana.
# Adaptar estos nombres según lo que realmente contenga el parquet.
# Patrones comunes: cluster_day, cluster_week, cluster_type_day, cluster_type_week

day_cluster_col = None
week_cluster_col = None

for c in cluster_cols:
    cl = c.lower()
    if 'day' in cl and 'week' not in cl:
        day_cluster_col = c
    elif 'week' in cl:
        week_cluster_col = c

# Respaldo: si solo hay una columna de clúster, usarla para ambos análisis
if day_cluster_col is None and week_cluster_col is None and len(cluster_cols) >= 1:
    day_cluster_col = cluster_cols[0]
    week_cluster_col = cluster_cols[1] if len(cluster_cols) > 1 else cluster_cols[0]

CLUSTER_TARGETS = {}
if day_cluster_col:
    CLUSTER_TARGETS['type_day'] = day_cluster_col
if week_cluster_col:
    CLUSTER_TARGETS['type_week'] = week_cluster_col

print(f"Objetivos de clúster: {CLUSTER_TARGETS}")
for name, col in CLUSTER_TARGETS.items():
    print(f"  {name} ({col}): {df[col].nunique()} clústeres, distribución:")
    print(f"    {df[col].value_counts().sort_index().to_dict()}")

In [ ]:
df

---

# 1. Análisis Clúster vs. Variables de Calendario

Evaluamos si las asignaciones de clúster son redundantes con las variables de calendario. Si kappa < 0.3 y NMI es cercano a cero, los clústeres codifican información **más allá** de lo que capturan las variables de calendario — por ejemplo, condiciones meteorológicas, actividad económica o patrones de comportamiento.

Si kappa > 0.5, los clústeres son parcialmente redundantes con las variables de calendario y pueden no aportar mucho valor adicional frente a un enrutamiento basado únicamente en calendario.

In [ ]:
# ---- Kappa de Cohen, NMI y Chi-cuadrado para cada clustering vs variables de calendario ----

calendar_vars = {
    'day_of_week': df['day_of_week'],
    'month': df['month'],
    'is_holiday': df['is_holiday'],
}

agreement_results = []

for clustering_name, cluster_col in CLUSTER_TARGETS.items():
    cluster_values = df[cluster_col]

    for cal_name, cal_values in calendar_vars.items():
        # Kappa de Cohen -- requiere el mismo número de categorías o tolera desajuste
        # Para variables con distinta cardinalidad, kappa sigue funcionando como medida
        kappa = cohen_kappa_score(cluster_values, cal_values)

        # Información mutua normalizada
        nmi = normalized_mutual_info_score(cluster_values, cal_values)

        # Prueba Chi-cuadrado de independencia
        contingency = pd.crosstab(cluster_values, cal_values)
        chi2, p_value, dof, expected = chi2_contingency(contingency)

        agreement_results.append({
            'clustering': clustering_name,
            'calendar_var': cal_name,
            'kappa': kappa,
            'nmi': nmi,
            'chi2': chi2,
            'chi2_p_value': p_value,
            'dof': dof,
        })

df_agreement = pd.DataFrame(agreement_results)
print("=" * 80)
print("RESUMEN DE ACUERDO ENTRE CLÚSTER Y VARIABLE DE CALENDARIO")
print("=" * 80)
display(df_agreement.style.format({
    'kappa': '{:.4f}',
    'nmi': '{:.4f}',
    'chi2': '{:.1f}',
    'chi2_p_value': '{:.2e}',
}).set_caption('Métricas de Acuerdo entre Clústeres y Variables de Calendario'))

In [ ]:
# ---- Guía de interpretación ----

print("\nGUÍA DE INTERPRETACIÓN:")
print("-" * 60)
print("Kappa de Cohen:")
print("  < 0.0  : Acuerdo menor al azar")
print("  0.0-0.2: Acuerdo leve")
print("  0.2-0.4: Acuerdo aceptable")
print("  0.4-0.6: Acuerdo moderado")
print("  0.6-0.8: Acuerdo sustancial")
print("  0.8-1.0: Acuerdo casi perfecto")
print("\nNMI (Información Mutua Normalizada):")
print("  0.0: Independientes (sin información compartida)")
print("  1.0: Correlación perfecta")
print("  < 0.1: Dependencia despreciable")
print("\nPrueba Chi-cuadrado:")
print("  p < 0.05: Dependencia estadística (se rechaza la independencia)")
print("  Nota: Con N grande, incluso efectos pequeños son significativos.")
print("  Enfocarse en el tamaño del efecto (kappa, NMI), no solo en el p-value.")

print("\n" + "=" * 60)
print("HALLAZGOS CLAVE:")
print("=" * 60)
for _, row in df_agreement.iterrows():
    kappa_interp = (
        "no redundante" if abs(row['kappa']) < 0.3
        else "parcialmente redundante" if abs(row['kappa']) < 0.5
        else "altamente redundante"
    )
    print(f"  {row['clustering']} vs {row['calendar_var']}: "
          f"kappa={row['kappa']:.3f} ({kappa_interp}), NMI={row['nmi']:.3f}")

In [ ]:
# ---- Matriz de confusión: cluster_id vs day_of_week (mapas de calor) ----

dow_labels = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(7 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, (clustering_name, cluster_col) in zip(axes, CLUSTER_TARGETS.items()):
    ct = pd.crosstab(
        df[cluster_col],
        df['day_of_week'],
        normalize='index',  # Normalizar por fila: distribución entre días PARA cada clúster
    )
    ct.columns = [dow_labels[i] for i in ct.columns]

    sns.heatmap(
        ct, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
        cbar_kws={'label': 'Proporción'},
        linewidths=0.5,
    )
    ax.set_title(f'{clustering_name}: Clúster vs Día de la Semana\n(normalizado por fila)', fontweight='bold')
    ax.set_xlabel('Día de la Semana')
    ax.set_ylabel('ID del Clúster')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_cluster_vs_dow_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Si las filas se ven uniformes, los clústeres NO corresponden a días específicos -- bueno para no redundancia.")

In [ ]:
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(7 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, (clustering_name, cluster_col) in zip(axes, CLUSTER_TARGETS.items()):
    ct = pd.crosstab(
        df[cluster_col],
        df['day_of_week'],
        normalize='index',  # Row-normalize: distribution across days FOR each cluster
    )
    ct.columns = [dow_labels[i] for i in ct.columns]

    sns.heatmap(
        ct, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
        cbar_kws={'label': 'Proportion'},
        linewidths=0.5,
    )
    ax.set_title(f'{clustering_name}: Cluster vs Day of Week\n(row-normalized)', fontweight='bold')
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Cluster ID')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "english", '04_cluster_vs_dow_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("If rows look uniform, clusters do NOT correspond to specific days -- good for non-redundancy.")

In [ ]:
# ---- Matriz de confusión: cluster_id vs month ----

month_labels = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(8 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, (clustering_name, cluster_col) in zip(axes, CLUSTER_TARGETS.items()):
    ct = pd.crosstab(
        df[cluster_col],
        df['month'],
        normalize='index',
    )
    ct.columns = [month_labels[i - 1] for i in ct.columns]

    sns.heatmap(
        ct, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax,
        cbar_kws={'label': 'Proporción'},
        linewidths=0.5,
    )
    ax.set_title(f'{clustering_name}: Clúster vs Mes\n(normalizado por fila)', fontweight='bold')
    ax.set_xlabel('Mes')
    ax.set_ylabel('ID del Clúster')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_cluster_vs_month_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Confusion Matrix: cluster_id vs month ----

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(8 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, (clustering_name, cluster_col) in zip(axes, CLUSTER_TARGETS.items()):
    ct = pd.crosstab(
        df[cluster_col],
        df['month'],
        normalize='index',
    )
    ct.columns = [month_labels[i - 1] for i in ct.columns]

    sns.heatmap(
        ct, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax,
        cbar_kws={'label': 'Proportion'},
        linewidths=0.5,
    )
    ax.set_title(f'{clustering_name}: Cluster vs Month\n(row-normalized)', fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Cluster ID')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "english", '04_cluster_vs_month_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Distribución de clústeres condicionada a cada variable de calendario ----

for clustering_name, cluster_col in CLUSTER_TARGETS.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{clustering_name}: Distribución de Clústeres Condicionada a Variables de Calendario',
                 fontweight='bold', fontsize=14, y=1.02)

    # (a) Condicionada por día de la semana
    ct_dow = pd.crosstab(df['day_of_week'], df[cluster_col], normalize='index')
    ct_dow.index = [dow_labels[i] for i in ct_dow.index]
    ct_dow.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Por Día de la Semana')
    axes[0].set_xlabel('Día de la Semana')
    axes[0].set_ylabel('Proporción de Clúster')
    axes[0].legend(title='Clúster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

    # (b) Condicionada por mes
    ct_month = pd.crosstab(df['month'], df[cluster_col], normalize='index')
    ct_month.index = [month_labels[i - 1] for i in ct_month.index]
    ct_month.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Por Mes')
    axes[1].set_xlabel('Mes')
    axes[1].set_ylabel('Proporción de Clúster')
    axes[1].legend(title='Clúster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

    # (c) Condicionada por festivo
    ct_hol = pd.crosstab(df['is_holiday'], df[cluster_col], normalize='index')
    ct_hol.index = ['No Festivo', 'Festivo']
    ct_hol.plot(kind='bar', stacked=True, ax=axes[2], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[2].set_title('Por Estado de Festivo')
    axes[2].set_xlabel('Festivo')
    axes[2].set_ylabel('Proporción de Clúster')
    axes[2].legend(title='Clúster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_cluster_dist_conditioned_{clustering_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Distribution of clusters conditioned on each calendar variable ----

for clustering_name, cluster_col in CLUSTER_TARGETS.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{clustering_name}: Cluster Distribution Conditioned on Calendar Variables',
                 fontweight='bold', fontsize=14, y=1.02)

    # (a) Conditioned on Day of Week
    ct_dow = pd.crosstab(df['day_of_week'], df[cluster_col], normalize='index')
    ct_dow.index = [dow_labels[i] for i in ct_dow.index]
    ct_dow.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[0].set_title('By Day of Week')
    axes[0].set_xlabel('Day of Week')
    axes[0].set_ylabel('Cluster Proportion')
    axes[0].legend(title='Cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

    # (b) Conditioned on Month
    ct_month = pd.crosstab(df['month'], df[cluster_col], normalize='index')
    ct_month.index = [month_labels[i - 1] for i in ct_month.index]
    ct_month.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[1].set_title('By Month')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Cluster Proportion')
    axes[1].legend(title='Cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

    # (c) Conditioned on Holiday
    ct_hol = pd.crosstab(df['is_holiday'], df[cluster_col], normalize='index')
    ct_hol.index = ['Non-Holiday', 'Holiday']
    ct_hol.plot(kind='bar', stacked=True, ax=axes[2], colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[2].set_title('By Holiday Status')
    axes[2].set_xlabel('Holiday')
    axes[2].set_ylabel('Cluster Proportion')
    axes[2].legend(title='Cluster', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'english/04_cluster_dist_conditioned_{clustering_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

---

# 2. Clasificadores con Múltiples Configuraciones de Conjuntos de Variables

Entrenamos clasificadores para predecir la pertenencia a clústeres usando conjuntos de variables progresivamente más ricos. Esto permite determinar qué combinación de variables y algoritmo es la más adecuada para el mecanismo de *gating*.

**Validación**: `TimeSeriesSplit(n_splits=5, gap=168)` -- el gap de 168 horas (una semana) evita fuga de información proveniente de variables de rezagos/ventanas móviles.

In [ ]:
# ---- Definir conjuntos de variables ----

# Identificar las columnas de variables disponibles mediante coincidencia de patrones
all_cols = list(df.columns)

calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

# Variables de Fourier
fourier_features = sorted([c for c in all_cols if c.startswith('fourier_')])

# Variables de rezagos
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

# Variables de rolling
rolling_features = sorted([c for c in all_cols if 'rolling' in c])

# Validar que todas las variables existan
for name, flist in [('calendar', calendar_only), ('fourier', fourier_features),
                     ('lag', lag_features), ('rolling', rolling_features)]:
    missing = [f for f in flist if f not in all_cols]
    if missing:
        print(f"ADVERTENCIA: Faltan variables de {name}: {missing}")
    else:
        print(f"{name}: {len(flist)} variables OK")

# Construir conjuntos de variables incrementales
feature_sets = {
    'calendar_only': calendar_only,
    'calendar_fourier': calendar_only + fourier_features,
    'calendar_lags': calendar_only + lag_features,
    'calendar_lags_rolling': calendar_only + lag_features + rolling_features,
    'all_production': calendar_only + fourier_features + lag_features + rolling_features,
}

for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} variables")

In [ ]:
# ---- Definir clasificadores ----
classifiers = {
    'LogisticRegression': LogisticRegression(
        max_iter=2000, multi_class='multinomial', random_state=RANDOM_SEED,
        C=1.0, n_jobs=N_JOBS
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        random_state=RANDOM_SEED, n_jobs=N_JOBS
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5,
        learning_rate=0.1, random_state=RANDOM_SEED
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric='mlogloss', verbosity=0, random_state=RANDOM_SEED),
    'KNeighbors': KNeighborsClassifier(
        n_neighbors=7, n_jobs=N_JOBS
    ),
    'SVC': SVC(
        kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_SEED
    ),
}

print(f"Clasificadores: {list(classifiers.keys())}")
print(f"Objetivos de clúster: {list(CLUSTER_TARGETS.keys())}")
print(f"Conjuntos de variables: {list(feature_sets.keys())}")
n_combos = len(classifiers) * len(feature_sets) * len(CLUSTER_TARGETS)
print(f"Combinaciones totales: {n_combos}")

In [ ]:
# ---- Ejecutar experimentos de clasificación con validación cruzada ----
tscv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

# Eliminar filas con NaN en cualquier variable (por rezagos/ventanas móviles al inicio de la serie)
all_feature_cols = feature_sets['all_production']
valid_mask = df[all_feature_cols].notna().all(axis=1)
df_valid = df[valid_mask].reset_index(drop=True)
print(f"Filas válidas después de eliminar NaN: {len(df_valid)} / {len(df)} ({len(df_valid)/len(df)*100:.1f}%)")

results = []

for target_name, cluster_col in CLUSTER_TARGETS.items():
    y = df_valid[cluster_col].values
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    n_classes = len(le.classes_)
    print(f"\n{'='*70}")
    print(f"OBJETIVO: {target_name} ({cluster_col}) -- {n_classes} clases")
    print(f"{'='*70}")

    for fs_name, fs_cols in feature_sets.items():
        # Verificar que todas las columnas existan en df_valid
        fs_cols_valid = [c for c in fs_cols if c in df_valid.columns]
        X = df_valid[fs_cols_valid].values

        for clf_name, clf_template in classifiers.items():
            fold_accs = []
            fold_f1s = []
            fold_probas = []  # Almacenar el máximo de predict_proba por fold

            for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X, y_encoded)):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

                # Pipeline con escalado (importante para LR, KNN, SVC)
                from sklearn.base import clone
                clf = clone(clf_template)
                pipe = Pipeline([
                    ('scaler', StandardScaler()),
                    ('clf', clf),
                ])

                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)

                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
                fold_accs.append(acc)
                fold_f1s.append(f1)

                # Confianza: máximo de predict_proba
                if hasattr(pipe.named_steps['clf'], 'predict_proba'):
                    probas = pipe.predict_proba(X_test)
                    max_proba = probas.max(axis=1)
                    fold_probas.extend(max_proba.tolist())

            mean_acc = np.mean(fold_accs)
            std_acc = np.std(fold_accs)
            mean_f1 = np.mean(fold_f1s)
            std_f1 = np.std(fold_f1s)
            mean_confidence = np.mean(fold_probas) if fold_probas else np.nan

            results.append({
                'target': target_name,
                'feature_set': fs_name,
                'classifier': clf_name,
                'accuracy_mean': mean_acc,
                'accuracy_std': std_acc,
                'f1_macro_mean': mean_f1,
                'f1_macro_std': std_f1,
                'mean_confidence': mean_confidence,
                'n_features': len(fs_cols_valid),
            })

            print(f"  {fs_name:25s} | {clf_name:20s} | "
                  f"Acc={mean_acc:.4f}+/-{std_acc:.4f} | "
                  f"F1={mean_f1:.4f}+/-{std_f1:.4f}")

df_results = pd.DataFrame(results)
print(f"\nTotal de experimentos: {len(df_results)}")

In [ ]:
# ---- Tabla resumen: feature_set x classifier -> métricas por objetivo ----

for target_name in CLUSTER_TARGETS:
    print(f"\n{'='*80}")
    print(f"TABLA RESUMEN: {target_name}")
    print(f"{'='*80}")

    subset = df_results[df_results['target'] == target_name]

    # Pivot para accuracy
    pivot_acc = subset.pivot_table(
        index='feature_set', columns='classifier',
        values='accuracy_mean', aggfunc='first'
    )
    pivot_f1 = subset.pivot_table(
        index='feature_set', columns='classifier',
        values='f1_macro_mean', aggfunc='first'
    )

    # Ordenar los conjuntos de variables de forma lógica
    fs_order = ['calendar_only', 'calendar_fourier', 'calendar_lags',
                'calendar_lags_rolling', 'all_production']
    fs_order = [f for f in fs_order if f in pivot_acc.index]

    print("\n--- Accuracy ---")
    display(pivot_acc.loc[fs_order].style.format('{:.4f}').background_gradient(
        cmap='YlGn', axis=None
    ).set_caption(f'Accuracy: {target_name}'))

    print("\n--- F1 Macro ---")
    display(pivot_f1.loc[fs_order].style.format('{:.4f}').background_gradient(
        cmap='YlGn', axis=None
    ).set_caption(f'F1 Macro: {target_name}'))

In [ ]:
# ---- Comparación visual: accuracy y F1 por conjunto de variables y clasificador ----

for target_name in CLUSTER_TARGETS:
    subset = df_results[df_results['target'] == target_name].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Comparación de Clasificadores: {target_name}', fontweight='bold', fontsize=14)

    fs_order = ['calendar_only', 'calendar_fourier', 'calendar_lags',
                'calendar_lags_rolling', 'all_production']
    subset['feature_set'] = pd.Categorical(subset['feature_set'], categories=fs_order, ordered=True)
    subset = subset.sort_values('feature_set')

    # Accuracy
    sns.barplot(
        data=subset, x='feature_set', y='accuracy_mean', hue='classifier',
        ax=axes[0], edgecolor='black', linewidth=0.5,
    )
    axes[0].set_title('Accuracy (promedio sobre folds de CV)')
    axes[0].set_xlabel('Conjunto de Variables')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
    axes[0].legend(title='Clasificador', fontsize=8, loc='lower right')

    # F1 Macro
    sns.barplot(
        data=subset, x='feature_set', y='f1_macro_mean', hue='classifier',
        ax=axes[1], edgecolor='black', linewidth=0.5,
    )
    axes[1].set_title('F1 Macro (promedio sobre folds de CV)')
    axes[1].set_xlabel('Conjunto de Variables')
    axes[1].set_ylabel('F1 Macro')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
    axes[1].legend(title='Clasificador', fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_classifier_comparison_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
for target_name in CLUSTER_TARGETS:
    subset = df_results[df_results['target'] == target_name].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Classifier Comparison: {target_name}', fontweight='bold', fontsize=14)

    fs_order = ['calendar_only', 'calendar_fourier', 'calendar_lags',
                'calendar_lags_rolling', 'all_production']
    subset['feature_set'] = pd.Categorical(subset['feature_set'], categories=fs_order, ordered=True)
    subset = subset.sort_values('feature_set')

    # Accuracy
    sns.barplot(
        data=subset, x='feature_set', y='accuracy_mean', hue='classifier',
        ax=axes[0], edgecolor='black', linewidth=0.5,
    )
    axes[0].set_title('Accuracy (mean over CV folds)')
    axes[0].set_xlabel('Feature Set')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
    axes[0].legend(title='Classifier', fontsize=8, loc='lower right')

    # F1 Macro
    sns.barplot(
        data=subset, x='feature_set', y='f1_macro_mean', hue='classifier',
        ax=axes[1], edgecolor='black', linewidth=0.5,
    )
    axes[1].set_title('F1 Macro (mean over CV folds)')
    axes[1].set_xlabel('Feature Set')
    axes[1].set_ylabel('F1 Macro')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
    axes[1].legend(title='Classifier', fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'english/04_classifier_comparison_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Identificar el mejor clasificador para cada objetivo ----

# Criterio de ranking: mayor F1 macro, luego accuracy como desempate
best_configs = {}

for target_name in CLUSTER_TARGETS:
    subset = df_results[df_results['target'] == target_name].copy()
    subset = subset.sort_values(
        ['f1_macro_mean', 'accuracy_mean'], ascending=[False, False]
    )
    best_row = subset.iloc[0]
    best_configs[target_name] = {
        'classifier': best_row['classifier'],
        'feature_set': best_row['feature_set'],
        'accuracy': best_row['accuracy_mean'],
        'f1_macro': best_row['f1_macro_mean'],
        'mean_confidence': best_row['mean_confidence'],
    }

    print(f"\nMEJOR para {target_name}:")
    print(f"  Clasificador: {best_row['classifier']}")
    print(f"  Conjunto de variables: {best_row['feature_set']}")
    print(f"  Accuracy: {best_row['accuracy_mean']:.4f} +/- {best_row['accuracy_std']:.4f}")
    print(f"  F1 Macro: {best_row['f1_macro_mean']:.4f} +/- {best_row['f1_macro_std']:.4f}")
    print(f"  Confianza media: {best_row['mean_confidence']:.4f}")

    # Mostrar top 5
    print(f"\n  Top 5 configuraciones:")
    display(subset.head(5)[[
        'feature_set', 'classifier', 'accuracy_mean', 'f1_macro_mean', 'mean_confidence'
    ]].reset_index(drop=True))

In [ ]:
# ---- Entrenar clasificadores finales óptimos sobre todos los datos ----
# (Estos se guardarán para uso en producción dentro del mecanismo de gating)

final_classifiers = {}
final_label_encoders = {}

for target_name, cluster_col in CLUSTER_TARGETS.items():
    config = best_configs[target_name]
    fs_cols = feature_sets[config['feature_set']]
    fs_cols_valid = [c for c in fs_cols if c in df_valid.columns]

    X_full = df_valid[fs_cols_valid].values
    y_full = df_valid[cluster_col].values
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_full)

    clf_template = classifiers[config['classifier']]
    from sklearn.base import clone
    clf = clone(clf_template)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', clf),
    ])
    pipe.fit(X_full, y_encoded)

    final_classifiers[target_name] = {
        'pipeline': pipe,
        'label_encoder': le,
        'feature_columns': fs_cols_valid,
        'config': config,
    }
    final_label_encoders[target_name] = le

    # Reporte de clasificación sobre todos los datos (desempeño en entrenamiento)
    y_pred_full = pipe.predict(X_full)
    print(f"\n{'='*60}")
    print(f"MODELO FINAL - {target_name} ({config['classifier']} + {config['feature_set']})")
    print(f"{'='*60}")
    print(classification_report(
        y_encoded, y_pred_full,
        target_names=[str(c) for c in le.classes_],
    ))

print("Clasificadores finales entrenados sobre todo el conjunto de datos.")

---

# 3. Análisis de Confianza del Clasificador

Entender cuándo el clasificador es incierto es crítico para el mecanismo de *gating*. Las predicciones con baja confianza indican observaciones que no pertenecen claramente a ningún clúster — típicamente festivos, días puente o transiciones estacionales.

Este análisis alimenta directamente el diseño de la red de *gating* (Notebook 06).

In [ ]:
# ---- Distribución de confianza para los clasificadores óptimos ----

confidence_analysis = {}

for target_name, model_info in final_classifiers.items():
    pipe = model_info['pipeline']
    fs_cols_valid = model_info['feature_columns']

    X_full = df_valid[fs_cols_valid].values
    probas = pipe.predict_proba(X_full)
    max_proba = probas.max(axis=1)

    df_valid[f'confidence_{target_name}'] = max_proba
    df_valid[f'predicted_{target_name}'] = pipe.predict(X_full)

    # Estadísticas
    conf_stats = {
        'mean': float(np.mean(max_proba)),
        'median': float(np.median(max_proba)),
        'std': float(np.std(max_proba)),
        'q05': float(np.percentile(max_proba, 5)),
        'q10': float(np.percentile(max_proba, 10)),
        'q25': float(np.percentile(max_proba, 25)),
        'q75': float(np.percentile(max_proba, 75)),
        'q90': float(np.percentile(max_proba, 90)),
        'pct_below_50': float((max_proba < 0.5).mean()),
        'pct_below_60': float((max_proba < 0.6).mean()),
        'pct_below_70': float((max_proba < 0.7).mean()),
        'pct_below_80': float((max_proba < 0.8).mean()),
    }
    confidence_analysis[target_name] = conf_stats

    print(f"\nEstadísticas de Confianza - {target_name}:")
    for k, v in conf_stats.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
# ---- Histograma de max(predict_proba) ----

fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(7 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, target_name in zip(axes, CLUSTER_TARGETS):
    conf_col = f'confidence_{target_name}'
    conf_values = df_valid[conf_col]

    ax.hist(conf_values, bins=50, edgecolor='black', linewidth=0.5, alpha=0.8, color='steelblue')

    # Marcar percentiles
    for pct, color, ls in [(10, 'red', '--'), (25, 'orange', '-.'), (50, 'green', ':')]:
        val = np.percentile(conf_values, pct)
        ax.axvline(val, color=color, linestyle=ls, linewidth=1.5, label=f'P{pct}={val:.2f}')

    ax.set_title(f'{target_name}: Distribución de max(predict_proba)', fontweight='bold')
    ax.set_xlabel('Probabilidad Máxima Predicha (Confianza)')
    ax.set_ylabel('Conteo')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_confidence_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(CLUSTER_TARGETS), figsize=(7 * len(CLUSTER_TARGETS), 5))
if len(CLUSTER_TARGETS) == 1:
    axes = [axes]

for ax, target_name in zip(axes, CLUSTER_TARGETS):
    conf_col = f'confidence_{target_name}'
    conf_values = df_valid[conf_col]

    ax.hist(conf_values, bins=50, edgecolor='black', linewidth=0.5, alpha=0.8, color='steelblue')

    # Mark percentiles
    for pct, color, ls in [(10, 'red', '--'), (25, 'orange', '-.'), (50, 'green', ':')]:
        val = np.percentile(conf_values, pct)
        ax.axvline(val, color=color, linestyle=ls, linewidth=1.5, label=f'P{pct}={val:.2f}')

    ax.set_title(f'{target_name}: Distribution of max(predict_proba)', fontweight='bold')
    ax.set_xlabel('Max Predicted Probability (Confidence)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'english/04_confidence_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Confianza por tipo de observación ----
# ¿Cuándo tiene baja confianza el clasificador? Se espera: festivos, días puente, transiciones.

for target_name in CLUSTER_TARGETS:
    conf_col = f'confidence_{target_name}'

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{target_name}: Análisis de Confianza por Tipo de Observación',
                 fontweight='bold', fontsize=14)

    # (a) Confianza por estado de festivo
    groups = df_valid.groupby('is_holiday')[conf_col]
    data_holiday = [groups.get_group(0).values, groups.get_group(1).values]
    bp = axes[0, 0].boxplot(data_holiday, labels=['No Festivo', 'Festivo'],
                            patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('salmon')
    axes[0, 0].set_title('Confianza por Estado de Festivo')
    axes[0, 0].set_ylabel('Probabilidad Máxima Predicha')

    # (b) Confianza por día de la semana
    dow_data = [df_valid[df_valid['day_of_week'] == d][conf_col].values for d in range(7)]
    bp2 = axes[0, 1].boxplot(dow_data, labels=dow_labels, patch_artist=True, widths=0.6)
    colors_dow = plt.cm.Set3(np.linspace(0, 1, 7))
    for box, color in zip(bp2['boxes'], colors_dow):
        box.set_facecolor(color)
    axes[0, 1].set_title('Confianza por Día de la Semana')
    axes[0, 1].set_ylabel('Probabilidad Máxima Predicha')
    axes[0, 1].set_xticklabels(dow_labels, rotation=45, ha='right')

    # (c) Confianza por estado de día puente
    if 'is_bridge' in df_valid.columns:
        groups_bridge = df_valid.groupby('is_bridge')[conf_col]
        data_bridge = [groups_bridge.get_group(0).values, groups_bridge.get_group(1).values]
        bp3 = axes[1, 0].boxplot(data_bridge, labels=['No Puente', 'Puente'],
                                  patch_artist=True, widths=0.5)
        bp3['boxes'][0].set_facecolor('lightblue')
        bp3['boxes'][1].set_facecolor('lightyellow')
        axes[1, 0].set_title('Confianza por Estado de Día Puente')
        axes[1, 0].set_ylabel('Probabilidad Máxima Predicha')

    # (d) Confianza promedio por hora del día
    hour_means = df_valid.groupby('hour')[conf_col].mean()
    hour_stds = df_valid.groupby('hour')[conf_col].std()
    axes[1, 1].fill_between(
        hour_means.index, hour_means - hour_stds, hour_means + hour_stds,
        alpha=0.3, color='steelblue',
    )
    axes[1, 1].plot(hour_means.index, hour_means, 'o-', color='steelblue', markersize=4)
    axes[1, 1].set_title('Confianza Media por Hora del Día')
    axes[1, 1].set_xlabel('Hora')
    axes[1, 1].set_ylabel('Probabilidad Máxima Media Predicha')
    axes[1, 1].set_xticks(range(0, 24, 2))

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_confidence_by_obs_type_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
for target_name in CLUSTER_TARGETS:
    conf_col = f'confidence_{target_name}'

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{target_name}: Confidence Analysis by Observation Type',
                 fontweight='bold', fontsize=14)

    # (a) Confidence by holiday status
    groups = df_valid.groupby('is_holiday')[conf_col]
    data_holiday = [groups.get_group(0).values, groups.get_group(1).values]
    bp = axes[0, 0].boxplot(data_holiday, labels=['Non-Holiday', 'Holiday'],
                            patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('salmon')
    axes[0, 0].set_title('Confidence by Holiday Status')
    axes[0, 0].set_ylabel('Max Predicted Probability')

    # (b) Confidence by day of week
    dow_data = [df_valid[df_valid['day_of_week'] == d][conf_col].values for d in range(7)]
    bp2 = axes[0, 1].boxplot(dow_data, labels=dow_labels, patch_artist=True, widths=0.6)
    colors_dow = plt.cm.Set3(np.linspace(0, 1, 7))
    for box, color in zip(bp2['boxes'], colors_dow):
        box.set_facecolor(color)
    axes[0, 1].set_title('Confidence by Day of Week')
    axes[0, 1].set_ylabel('Max Predicted Probability')
    axes[0, 1].set_xticklabels(dow_labels, rotation=45, ha='right')

    # (c) Confidence by bridge status
    if 'is_bridge' in df_valid.columns:
        groups_bridge = df_valid.groupby('is_bridge')[conf_col]
        data_bridge = [groups_bridge.get_group(0).values, groups_bridge.get_group(1).values]
        bp3 = axes[1, 0].boxplot(data_bridge, labels=['Non-Bridge', 'Bridge'],
                                  patch_artist=True, widths=0.5)
        bp3['boxes'][0].set_facecolor('lightblue')
        bp3['boxes'][1].set_facecolor('lightyellow')
        axes[1, 0].set_title('Confidence by Bridge Day Status')
        axes[1, 0].set_ylabel('Max Predicted Probability')

    # (d) Confidence by hour of day
    hour_means = df_valid.groupby('hour')[conf_col].mean()
    hour_stds = df_valid.groupby('hour')[conf_col].std()
    axes[1, 1].fill_between(
        hour_means.index, hour_means - hour_stds, hour_means + hour_stds,
        alpha=0.3, color='steelblue',
    )
    axes[1, 1].plot(hour_means.index, hour_means, 'o-', color='steelblue', markersize=4)
    axes[1, 1].set_title('Mean Confidence by Hour of Day')
    axes[1, 1].set_xlabel('Hour')
    axes[1, 1].set_ylabel('Mean Max Predicted Probability')
    axes[1, 1].set_xticks(range(0, 24, 2))

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'english/04_confidence_by_obs_type_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Curvas de calibración y diagramas de confiabilidad ----

for target_name, model_info in final_classifiers.items():
    pipe = model_info['pipeline']
    le = model_info['label_encoder']
    fs_cols_valid = model_info['feature_columns']
    cluster_col = CLUSTER_TARGETS[target_name]

    X_full = df_valid[fs_cols_valid].values
    y_true = le.transform(df_valid[cluster_col].values)
    probas = pipe.predict_proba(X_full)
    n_classes = probas.shape[1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'{target_name}: Análisis de Calibración', fontweight='bold', fontsize=14)

    # (a) Curvas de calibración por clase
    for cls_idx in range(n_classes):
        y_binary = (y_true == cls_idx).astype(int)
        prob_cls = probas[:, cls_idx]

        if y_binary.sum() < 10:
            continue

        fraction_pos, mean_predicted = calibration_curve(
            y_binary, prob_cls, n_bins=10, strategy='uniform'
        )
        axes[0].plot(mean_predicted, fraction_pos, 's-',
                     label=f'Clúster {le.classes_[cls_idx]}', markersize=4)

    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibración perfecta')
    axes[0].set_title('Curva de Calibración (por clase)')
    axes[0].set_xlabel('Probabilidad Media Predicha')
    axes[0].set_ylabel('Fracción de Positivos')
    axes[0].legend(fontsize=8)
    axes[0].set_xlim([-0.02, 1.02])
    axes[0].set_ylim([-0.02, 1.02])

    # (b) Diagrama de confiabilidad: confianza vs exactitud real
    max_proba = probas.max(axis=1)
    y_pred = probas.argmax(axis=1)
    correct = (y_pred == y_true).astype(int)

    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_accs = []
    bin_counts = []

    for i in range(n_bins):
        mask = (max_proba >= bin_edges[i]) & (max_proba < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_accs.append(correct[mask].mean())
            bin_counts.append(mask.sum())

    ax_bar = axes[1]
    ax_bar2 = ax_bar.twinx()

    bars = ax_bar.bar(bin_centers, bin_accs, width=0.08, alpha=0.7,
                      color='steelblue', edgecolor='black', linewidth=0.5, label='Accuracy')
    ax_bar.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibración perfecta')
    ax_bar.set_title('Diagrama de Confiabilidad')
    ax_bar.set_xlabel('Confianza Media (probabilidad máxima)')
    ax_bar.set_ylabel('Exactitud Real', color='steelblue')
    ax_bar.set_xlim([-0.02, 1.02])
    ax_bar.set_ylim([-0.02, 1.02])
    ax_bar.legend(loc='upper left', fontsize=8)

    ax_bar2.bar(bin_centers, bin_counts, width=0.04, alpha=0.4,
                color='orange', edgecolor='black', linewidth=0.3, label='Conteo')
    ax_bar2.set_ylabel('Cantidad de Muestras', color='orange')
    ax_bar2.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_calibration_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Calibration curves and reliability diagrams ----

for target_name, model_info in final_classifiers.items():
    pipe = model_info['pipeline']
    le = model_info['label_encoder']
    fs_cols_valid = model_info['feature_columns']
    cluster_col = CLUSTER_TARGETS[target_name]

    X_full = df_valid[fs_cols_valid].values
    y_true = le.transform(df_valid[cluster_col].values)
    probas = pipe.predict_proba(X_full)
    n_classes = probas.shape[1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'{target_name}: Calibration Analysis', fontweight='bold', fontsize=14)

    # (a) Per-class calibration curves
    for cls_idx in range(n_classes):
        y_binary = (y_true == cls_idx).astype(int)
        prob_cls = probas[:, cls_idx]

        if y_binary.sum() < 10:
            continue

        fraction_pos, mean_predicted = calibration_curve(
            y_binary, prob_cls, n_bins=10, strategy='uniform'
        )
        axes[0].plot(mean_predicted, fraction_pos, 's-',
                     label=f'Cluster {le.classes_[cls_idx]}', markersize=4)

    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    axes[0].set_title('Calibration Curve (per class)')
    axes[0].set_xlabel('Mean Predicted Probability')
    axes[0].set_ylabel('Fraction of Positives')
    axes[0].legend(fontsize=8)
    axes[0].set_xlim([-0.02, 1.02])
    axes[0].set_ylim([-0.02, 1.02])

    # (b) Reliability diagram: confidence vs actual accuracy
    max_proba = probas.max(axis=1)
    y_pred = probas.argmax(axis=1)
    correct = (y_pred == y_true).astype(int)

    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_accs = []
    bin_counts = []

    for i in range(n_bins):
        mask = (max_proba >= bin_edges[i]) & (max_proba < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_accs.append(correct[mask].mean())
            bin_counts.append(mask.sum())

    ax_bar = axes[1]
    ax_bar2 = ax_bar.twinx()

    bars = ax_bar.bar(bin_centers, bin_accs, width=0.08, alpha=0.7,
                      color='steelblue', edgecolor='black', linewidth=0.5, label='Accuracy')
    ax_bar.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax_bar.set_title('Reliability Diagram')
    ax_bar.set_xlabel('Mean Confidence (max probability)')
    ax_bar.set_ylabel('Actual Accuracy', color='steelblue')
    ax_bar.set_xlim([-0.02, 1.02])
    ax_bar.set_ylim([-0.02, 1.02])
    ax_bar.legend(loc='upper left', fontsize=8)

    ax_bar2.bar(bin_centers, bin_counts, width=0.04, alpha=0.4,
                color='orange', edgecolor='black', linewidth=0.3, label='Count')
    ax_bar2.set_ylabel('Sample Count', color='orange')
    ax_bar2.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'english/04_calibration_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Sugerir umbral de confianza ----
# Umbral natural: el punto donde la exactitud cae significativamente.
# Buscamos el nivel de confianza por debajo del cual la precisión de clasificación disminuye notablemente.

for target_name in CLUSTER_TARGETS:
    conf_col = f'confidence_{target_name}'
    pred_col = f'predicted_{target_name}'
    cluster_col = CLUSTER_TARGETS[target_name]

    le = final_classifiers[target_name]['label_encoder']
    y_true = le.transform(df_valid[cluster_col].values)
    y_pred = df_valid[pred_col].values
    conf_values = df_valid[conf_col].values
    correct = (y_pred == y_true)

    thresholds = np.arange(0.3, 0.95, 0.05)
    results_thresh = []
    for t in thresholds:
        mask_above = conf_values >= t
        if mask_above.sum() > 0:
            acc_above = correct[mask_above].mean()
            pct_above = mask_above.mean()
        else:
            acc_above = np.nan
            pct_above = 0.0
        results_thresh.append({
            'threshold': t,
            'accuracy_above': acc_above,
            'pct_retained': pct_above,
        })

    df_thresh = pd.DataFrame(results_thresh)

    # Umbral sugerido: retener al menos 90% de los datos con la mejor exactitud
    viable = df_thresh[df_thresh['pct_retained'] >= 0.90]
    if len(viable) > 0:
        best_thresh = viable.loc[viable['accuracy_above'].idxmax(), 'threshold']
    else:
        best_thresh = df_thresh.loc[df_thresh['pct_retained'].idxmax(), 'threshold']

    confidence_analysis[target_name]['suggested_threshold'] = float(best_thresh)

    print(f"\n{target_name}: Análisis de Umbral")
    print(f"  Umbral sugerido: {best_thresh:.2f}")
    display(df_thresh.style.format({
        'threshold': '{:.2f}',
        'accuracy_above': '{:.4f}',
        'pct_retained': '{:.4f}',
    }).set_caption(f'{target_name}: Exactitud vs Umbral de Confianza'))

    # Gráfico
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    l1 = ax1.plot(df_thresh['threshold'], df_thresh['accuracy_above'], 'o-',
                  color='steelblue', label='Accuracy (sobre umbral)', markersize=5)
    l2 = ax2.plot(df_thresh['threshold'], df_thresh['pct_retained'], 's-',
                  color='coral', label='% Retenido', markersize=5)

    ax1.axvline(best_thresh, color='green', linestyle='--', linewidth=1.5,
                label=f'Sugerido: {best_thresh:.2f}')

    ax1.set_xlabel('Umbral de Confianza')
    ax1.set_ylabel('Exactitud', color='steelblue')
    ax2.set_ylabel('% Datos Retenidos', color='coral')
    ax1.set_title(f'{target_name}: Exactitud vs Umbral de Confianza', fontweight='bold')

    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='lower left', fontsize=9)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_threshold_analysis_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

In [ ]:
# ---- Suggest confidence threshold ----
# Natural threshold: the point where accuracy drops significantly.
# We look for the confidence level below which classification accuracy drops notably.

for target_name in CLUSTER_TARGETS:
    conf_col = f'confidence_{target_name}'
    pred_col = f'predicted_{target_name}'
    cluster_col = CLUSTER_TARGETS[target_name]

    le = final_classifiers[target_name]['label_encoder']
    y_true = le.transform(df_valid[cluster_col].values)
    y_pred = df_valid[pred_col].values
    conf_values = df_valid[conf_col].values
    correct = (y_pred == y_true)

    thresholds = np.arange(0.3, 0.95, 0.05)
    results_thresh = []
    for t in thresholds:
        mask_above = conf_values >= t
        if mask_above.sum() > 0:
            acc_above = correct[mask_above].mean()
            pct_above = mask_above.mean()
        else:
            acc_above = np.nan
            pct_above = 0.0
        results_thresh.append({
            'threshold': t,
            'accuracy_above': acc_above,
            'pct_retained': pct_above,
        })

    df_thresh = pd.DataFrame(results_thresh)

    # Suggested threshold: retain at least 90% of data with best accuracy
    viable = df_thresh[df_thresh['pct_retained'] >= 0.90]
    if len(viable) > 0:
        best_thresh = viable.loc[viable['accuracy_above'].idxmax(), 'threshold']
    else:
        best_thresh = df_thresh.loc[df_thresh['pct_retained'].idxmax(), 'threshold']

    confidence_analysis[target_name]['suggested_threshold'] = float(best_thresh)

    print(f"\n{target_name}: Threshold Analysis")
    print(f"  Suggested threshold: {best_thresh:.2f}")
    display(df_thresh.style.format({
        'threshold': '{:.2f}',
        'accuracy_above': '{:.4f}',
        'pct_retained': '{:.4f}',
    }).set_caption(f'{target_name}: Accuracy vs Confidence Threshold'))

    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    l1 = ax1.plot(df_thresh['threshold'], df_thresh['accuracy_above'], 'o-',
                  color='steelblue', label='Accuracy (above threshold)', markersize=5)
    l2 = ax2.plot(df_thresh['threshold'], df_thresh['pct_retained'], 's-',
                  color='coral', label='% Retained', markersize=5)

    ax1.axvline(best_thresh, color='green', linestyle='--', linewidth=1.5,
                label=f'Suggested: {best_thresh:.2f}')

    ax1.set_xlabel('Confidence Threshold')
    ax1.set_ylabel('Accuracy', color='steelblue')
    ax2.set_ylabel('% Data Retained', color='coral')
    ax1.set_title(f'{target_name}: Accuracy vs. Confidence Threshold', fontweight='bold')

    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='lower left', fontsize=9)

    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'04_threshold_analysis_{target_name}.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

---

# 4. Guardar Archivos

Persistimos los clasificadores óptimos (para uso en el mecanismo de *gating* en el Notebook 06), la tabla completa de comparación y los metadatos del análisis de confianza.

In [ ]:
# ---- Guardar clasificadores ----

classifiers_dir = os.path.join(MODELS_DIR, 'classifiers')
os.makedirs(classifiers_dir, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

for target_name, model_info in final_classifiers.items():
    # Determinar nombre de archivo
    if 'day' in target_name:
        fname = 'best_classifier_day.pkl'
    else:
        fname = 'best_classifier_week.pkl'

    save_path = os.path.join(classifiers_dir, fname)

    artifact = {
        'pipeline': model_info['pipeline'],
        'label_encoder': model_info['label_encoder'],
        'feature_columns': model_info['feature_columns'],
        'config': model_info['config'],
        'cluster_column': CLUSTER_TARGETS[target_name],
        'target_name': target_name,
    }

    with open(save_path, 'wb') as f:
        pickle.dump(artifact, f)

    print(f"Guardado: {save_path}")

# ---- Guardar tabla de comparación ----
results_csv_path = os.path.join(RESULTS_DIR, '04_classifier_comparison.csv')
df_results.to_csv(results_csv_path, index=False)
print(f"Guardado: {results_csv_path}")

# ---- Guardar análisis de confianza ----
conf_json_path = os.path.join(RESULTS_DIR, '04_confidence_analysis.json')

# Añadir información del mejor clasificador
confidence_output = {
    'analysis': confidence_analysis,
    'best_classifiers': best_configs,
}

with open(conf_json_path, 'w') as f:
    json.dump(confidence_output, f, indent=2)
print(f"Guardado: {conf_json_path}")

print("\nTodos los archivos guardados exitosamente.")